# 09 - Signal Model Evaluation

Load all trained signal models and evaluate on held-out test sets.

## Target Metrics
- **MAE < 10** per signal (on 0-100 scale)
- **Pearson r > 0.7** per signal

## Models Evaluated
1. **Structure** (05) - MobileNetV3, pore + texture + structure
2. **Hydration** (06) - EfficientNet-B0, Gabor + LBP hydration
3. **Elasticity** (07) - EfficientNet-B0, Frangi wrinkle
4. **Lesion Detection** (08) - YOLOv8s, mAP metrics

## Outputs
- Per-signal metrics table
- Scatter plots (predicted vs actual)
- Confusion matrix for lesion detection
- Combined metrics report JSON

In [ ]:
# Install dependencies (Colab)
!pip install -q torch torchvision timm onnxruntime ultralytics opencv-python scikit-learn scikit-image matplotlib seaborn

In [ ]:
import json
import numpy as np
import torch
import onnxruntime as ort
from pathlib import Path
from scipy import stats
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Signal model paths (update to your trained model locations)
MODEL_PATHS = {
    "structure": "structure_model.onnx",
    "hydration": "hydration_model.onnx",
    "elasticity": "elasticity_model.onnx",
    "lesion": "runs/lesion/phase2_acne/weights/best.pt",
}

TEST_DATA_DIR = Path("./data/test")

# Target thresholds
TARGET_MAE = 10.0
TARGET_PEARSON = 0.7

## Evaluation Utilities

In [ ]:
def compute_regression_metrics(predictions: np.ndarray, targets: np.ndarray) -> dict:
    """Compute MAE, RMSE, Pearson r, and Spearman rho for regression outputs."""
    mae = np.abs(predictions - targets).mean()
    rmse = np.sqrt(((predictions - targets) ** 2).mean())
    pearson_r, pearson_p = stats.pearsonr(predictions, targets)
    spearman_rho, spearman_p = stats.spearmanr(predictions, targets)
    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "pearson_r": float(pearson_r),
        "pearson_p": float(pearson_p),
        "spearman_rho": float(spearman_rho),
        "spearman_p": float(spearman_p),
        "passes_mae": bool(mae < TARGET_MAE),
        "passes_pearson": bool(pearson_r > TARGET_PEARSON),
    }


def plot_scatter(predictions, targets, signal_name, metrics):
    """Plot predicted vs actual scatter with regression line."""
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.scatter(targets, predictions, alpha=0.4, s=10, c="#7DE7E1")

    # Regression line
    z = np.polyfit(targets, predictions, 1)
    p = np.poly1d(z)
    x_range = np.linspace(targets.min(), targets.max(), 100)
    ax.plot(x_range, p(x_range), "r--", alpha=0.8, label="Regression line")

    # Perfect prediction line
    ax.plot([0, 100], [0, 100], "k--", alpha=0.3, label="Perfect prediction")

    ax.set_xlabel("Actual Score")
    ax.set_ylabel("Predicted Score")
    ax.set_title(f"{signal_name}\nMAE={metrics['mae']:.2f}, r={metrics['pearson_r']:.3f}")
    ax.legend()
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_aspect("equal")
    plt.tight_layout()
    return fig


def plot_error_distribution(predictions, targets, signal_name):
    """Plot error distribution histogram."""
    errors = predictions - targets
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    ax.hist(errors, bins=50, color="#7DE7E1", alpha=0.7, edgecolor="white")
    ax.axvline(x=0, color="red", linestyle="--", alpha=0.5)
    ax.set_xlabel("Prediction Error")
    ax.set_ylabel("Count")
    ax.set_title(f"{signal_name} - Error Distribution")
    plt.tight_layout()
    return fig

## Evaluate Regression Signal Models (Structure, Hydration, Elasticity)

In [ ]:
def evaluate_onnx_signal(model_path: str, test_images: np.ndarray, test_labels: np.ndarray,
                         signal_name: str, handcrafted_features: np.ndarray = None) -> dict:
    """Evaluate an ONNX signal model on test data."""
    session = ort.InferenceSession(model_path)
    input_names = [inp.name for inp in session.get_inputs()]

    # Run inference
    if handcrafted_features is not None and len(input_names) > 1:
        feed = {input_names[0]: test_images, input_names[1]: handcrafted_features}
    else:
        feed = {input_names[0]: test_images}

    outputs = session.run(None, feed)
    predictions = outputs[0].squeeze()

    # For structure model, evaluate each sub-task
    if predictions.ndim == 2 and predictions.shape[1] == 3:
        sub_metrics = {}
        sub_names = ["pore_count", "texture_regularity", "structure_score"]
        for i, name in enumerate(sub_names):
            sub_metrics[name] = compute_regression_metrics(predictions[:, i], test_labels[:, i])
        # Overall uses structure_score
        metrics = sub_metrics["structure_score"]
        metrics["sub_metrics"] = sub_metrics
    else:
        metrics = compute_regression_metrics(predictions, test_labels.squeeze())

    print(f"\n{'='*50}")
    print(f"{signal_name} Signal Evaluation")
    print(f"{'='*50}")
    print(f"  MAE:       {metrics['mae']:.2f} {'PASS' if metrics['passes_mae'] else 'FAIL'} (target < {TARGET_MAE})")
    print(f"  RMSE:      {metrics['rmse']:.2f}")
    print(f"  Pearson r: {metrics['pearson_r']:.4f} {'PASS' if metrics['passes_pearson'] else 'FAIL'} (target > {TARGET_PEARSON})")
    print(f"  Spearman:  {metrics['spearman_rho']:.4f}")

    return metrics


# Example evaluation (uncomment when models and test data are available)
# N_TEST = 500
# dummy_images = np.random.randn(N_TEST, 3, 224, 224).astype(np.float32)
# dummy_labels = np.random.uniform(20, 90, N_TEST).astype(np.float32)

# structure_metrics = evaluate_onnx_signal(
#     MODEL_PATHS["structure"], dummy_images,
#     np.column_stack([dummy_labels, dummy_labels, dummy_labels]),
#     "Structure"
# )

# dummy_hc_hydration = np.random.randn(N_TEST, 44).astype(np.float32)
# hydration_metrics = evaluate_onnx_signal(
#     MODEL_PATHS["hydration"], dummy_images, dummy_labels,
#     "Hydration", handcrafted_features=dummy_hc_hydration
# )

# dummy_hc_elasticity = np.random.randn(N_TEST, 14).astype(np.float32)
# elasticity_metrics = evaluate_onnx_signal(
#     MODEL_PATHS["elasticity"], dummy_images, dummy_labels,
#     "Elasticity", handcrafted_features=dummy_hc_elasticity
# )

## Evaluate Lesion Detection (YOLOv8)

In [ ]:
LESION_CLASSES = ["comedone", "papule", "pustule", "nodule", "macule", "patch"]


def evaluate_lesion_model(model_path: str, test_yaml: str) -> dict:
    """Evaluate YOLOv8 lesion detection model."""
    from ultralytics import YOLO
    model = YOLO(model_path)
    results = model.val(data=test_yaml, imgsz=640, batch=16)

    metrics = {
        "mAP50": float(results.box.map50),
        "mAP50_95": float(results.box.map),
        "precision": float(results.box.mp),
        "recall": float(results.box.mr),
        "per_class_ap50": {},
    }

    print(f"\n{'='*50}")
    print(f"Lesion Detection Evaluation")
    print(f"{'='*50}")
    print(f"  mAP@0.5:     {metrics['mAP50']:.4f}")
    print(f"  mAP@0.5:0.95: {metrics['mAP50_95']:.4f}")
    print(f"  Precision:   {metrics['precision']:.4f}")
    print(f"  Recall:      {metrics['recall']:.4f}")
    print(f"\n  Per-class AP@0.5:")
    for i, cls_name in enumerate(LESION_CLASSES):
        if i < len(results.box.ap50):
            ap = float(results.box.ap50[i])
            metrics["per_class_ap50"][cls_name] = ap
            print(f"    {cls_name:12s}: {ap:.4f}")

    return metrics


def plot_confusion_matrix(y_true, y_pred, class_names):
    """Plot confusion matrix for lesion detection."""
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names,
                yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Lesion Detection Confusion Matrix")
    plt.tight_layout()
    return fig


# Uncomment to evaluate
# lesion_metrics = evaluate_lesion_model(MODEL_PATHS["lesion"], str(TEST_DATA_DIR / "dataset.yaml"))

## Combined Metrics Report

In [ ]:
def generate_report(signal_metrics: dict, lesion_metrics: dict, output_path: str = "evaluation_report.json"):
    """Generate combined metrics report across all models."""
    report = {
        "target_thresholds": {
            "regression_mae": TARGET_MAE,
            "regression_pearson_r": TARGET_PEARSON,
        },
        "signal_models": {},
        "lesion_detection": lesion_metrics,
        "summary": {"total_models": 0, "passing": 0, "failing": 0},
    }

    for signal_name, metrics in signal_metrics.items():
        passes = metrics.get("passes_mae", False) and metrics.get("passes_pearson", False)
        report["signal_models"][signal_name] = {
            "mae": metrics["mae"],
            "pearson_r": metrics["pearson_r"],
            "passes": passes,
        }
        report["summary"]["total_models"] += 1
        if passes:
            report["summary"]["passing"] += 1
        else:
            report["summary"]["failing"] += 1

    # Lesion model pass/fail based on mAP
    if lesion_metrics:
        report["summary"]["total_models"] += 1
        if lesion_metrics.get("mAP50", 0) > 0.5:
            report["summary"]["passing"] += 1
        else:
            report["summary"]["failing"] += 1

    with open(output_path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\nReport saved to {output_path}")

    # Summary table
    print(f"\n{'='*60}")
    print(f"{'Model':<20} {'MAE':<10} {'Pearson r':<12} {'Status':<10}")
    print(f"{'-'*60}")
    for name, m in report["signal_models"].items():
        status = "PASS" if m["passes"] else "FAIL"
        print(f"{name:<20} {m['mae']:<10.2f} {m['pearson_r']:<12.4f} {status:<10}")
    if lesion_metrics:
        ld_status = "PASS" if lesion_metrics.get("mAP50", 0) > 0.5 else "FAIL"
        print(f"{'lesion_detection':<20} {'mAP50=' + f'{lesion_metrics.get("mAP50", 0):.2f}':<10} {'':<12} {ld_status:<10}")
    print(f"{'='*60}")
    s = report["summary"]
    print(f"Total: {s['passing']}/{s['total_models']} models passing")

    return report


# Uncomment when all evaluations have run
# all_signal_metrics = {
#     "structure": structure_metrics,
#     "hydration": hydration_metrics,
#     "elasticity": elasticity_metrics,
# }
# report = generate_report(all_signal_metrics, lesion_metrics)

## Visualization: All Signals

In [ ]:
def plot_all_signals(signal_results: dict):
    """Generate scatter plots for all signal models in a grid."""
    signals = list(signal_results.keys())
    n = len(signals)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
    if n == 1:
        axes = [axes]

    colors = {"structure": "#7DE7E1", "hydration": "#4DA6FF", "elasticity": "#B68AFF"}

    for ax, signal_name in zip(axes, signals):
        result = signal_results[signal_name]
        preds = result["predictions"]
        targets = result["targets"]
        metrics = result["metrics"]
        color = colors.get(signal_name, "#7DE7E1")

        ax.scatter(targets, preds, alpha=0.4, s=10, c=color)
        ax.plot([0, 100], [0, 100], "k--", alpha=0.3)
        z = np.polyfit(targets, preds, 1)
        p = np.poly1d(z)
        x_range = np.linspace(0, 100, 100)
        ax.plot(x_range, p(x_range), "r--", alpha=0.8)
        ax.set_xlabel("Actual")
        ax.set_ylabel("Predicted")
        ax.set_title(f"{signal_name.title()}\nMAE={metrics['mae']:.2f}, r={metrics['pearson_r']:.3f}")
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_aspect("equal")

    plt.suptitle("Signal Model Evaluation - Predicted vs Actual", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig("signal_evaluation_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved scatter plot to signal_evaluation_scatter.png")


# Uncomment when results are available
# plot_all_signals(signal_results)